# فصل ۱۱: خطا، اشکال‌زدایی و آزمون
کد و خروجی‌ها انگلیسی‌اند. هر مثال ورودی مستقل دارد. Run All را در کرنل تازه اجرا کنید.

## C01 — خطای نحوی، پیش از اجرای بدنه
وقتی یک دونقطه جا افتاده است، مفسر هنوز به اجرای آن قطعه نرسیده است. برای آنکه دفترچهٔ سالم متوقف نشود، متن ناسالم را به compile می‌دهیم؛ خودِ سلول از نظر نحوی معتبر است.

In [ ]:
try:
    compile("if True print('ready')", "broken_sample.py", "exec")
except SyntaxError as exc:
    print(type(exc).__name__)
    print(exc.lineno)

SyntaxError می‌تواند هنگام import یا compile نیز ایجاد شود. خط و پیکان محل تشخیص‌اند، نه تضمین اینکه علت دقیقاً همان نویسه است؛ پرانتز باز در خط قبل هم می‌تواند عامل باشد.

## C02 — دسته‌بندی خطای زمان اجرا
سه عملیات ظاهراً ساده، سه قرارداد متفاوت را نقض می‌کنند. نوع استثنا اطلاعات اولیهٔ خوبی دربارهٔ مسئله می‌دهد، اما جای خواندن زمینه را نمی‌گیرد.

In [ ]:
actions = [lambda: int("two"), lambda: 1 / 0, lambda: {}["price"]]
for action in actions:
    try:
        action()
    except (ValueError, ZeroDivisionError, KeyError) as exc:
        print(type(exc).__name__)

این tuple فقط برای نمایش چند نوع خطای شناخته‌شده است. در برنامهٔ واقعی ممکن است واکنش مناسب به هر نوع متفاوت باشد. هیچ‌یک از این خطاها را صرفاً برای ادامهٔ برنامه به صفر تبدیل نمی‌کنیم.

## C03 — خطای منطقی بدون هیچ استثنا
سفارش سه قلم کالا با قیمت ۱۰۰ و یک قلم برگشتی را در نظر بگیرید. پاسخ دستی ۲۰۰ است؛ کم‌کردن تعداد از مبلغ از نظر واحد نیز نادرست است.

In [ ]:
def wrong_retained(quantity, price, returned):
    return quantity * price - returned

actual = wrong_retained(3, 100, 1)
print(actual)
print(actual == 200)

اجرای موفق فقط ثابت می‌کند برنامه به مانع اجرایی نخورده است. مشاهدهٔ ۲۹۹ شاهد خطای فرمول است؛ آزمون مرجع و بررسی واحدها آن را آشکار می‌کند. این تابع عمداً غلط است و وارد پروژهٔ سالم نمی‌شود.

## C04 — خواندن مسیر فراخوانی
برای حذف تفاوت مسیر و شماره‌خط محیط‌ها، تنها نام دو قاب آخر و نوع خطا را چاپ می‌کنیم. traceback کامل برای تشخیص واقعی حفظ می‌شود.

In [ ]:
import traceback

def divide(total, count):
    return total / count

def report():
    return divide(100, 0)

try:
    report()
except ZeroDivisionError as exc:
    frames = traceback.extract_tb(exc.__traceback__)
    print([frame.name for frame in frames[-2:]])
    print(type(exc).__name__)

آخرین قاب معمولاً محل وقوع مستقیم است؛ قاب قبلی می‌گوید ورودی نامعتبر از کجا آمده است. در گزارش زنجیره‌ای، بخش علت و بخش استثنای بیرونی را جدا بخوانید. شماره‌خط‌ها پس از ویرایش می‌توانند تغییر کنند.

## C05 — مشاهدهٔ مقدار با repr و type
دو مقدار ممکن است روی صفحه شبیه هم به نظر برسند، اما فاصلهٔ پنهان یا نوعشان فرق کند. پیش از هر تبدیل، فرضیه را با مشاهدهٔ محدود بررسی کنید.

In [ ]:
value = " 12 "
print(repr(value))
print(type(value).__name__)
print(len(value))

repr برای دیدن فاصله و escape مناسب است، ولی مجوز چاپ دادهٔ حساس نیست. در پروژهٔ این فصل، پیام عمومی فقط شمارهٔ رکورد، نام فیلد و کد خطا دارد؛ متن خام مشتری یا مسیر در پیام عادی افشا نمی‌شود.

## C06 — بازتولید کمینه و اصلاح فرمول
پیش از سراغ گرفتن کل CSV، خطای فرمول را با یک تابع و سه عدد بازتولید کنید. کوچک‌کردن ورودی باید ویژگی مولد خطا را حفظ کند.

In [ ]:
def retained(quantity, price, returned):
    return (quantity - returned) * price

print(retained(3, 100, 1))
print(retained(3, 100, 0))
print(retained(3, 100, 3))

سه حالت عادی، بدون برگشت و برگشت کامل، سه شاهد متمایزند. هنوز ورودی منفی یا تعداد اعشاری را کنترل نکرده‌ایم؛ مرز مسئولیت تابع باید صریح باشد، نه اینکه از نتیجهٔ این سه اجرا اعتبار همهٔ ورودی‌ها را نتیجه بگیریم.

## C07 — گرفتن استثنای مشخص
تبدیل متن نامعتبر به عدد یک شکست قابل‌انتظار است. محدودهٔ try را کوچک نگه می‌داریم تا خطای بخش دیگری را اشتباه به کاربر نسبت ندهیم.

In [ ]:
def parse_count(text):
    try:
        return int(text)
    except ValueError:
        return None

print(parse_count("12"))
print(parse_count("twelve"))

None در این تابع یک قرارداد صریح برای شکست تبدیل است، نه مقدار صفر. TypeError عمداً گرفته نمی‌شود؛ اگر فراخواننده list بفرستد، نقص قرارداد نباید پنهان شود. در پروژهٔ اصلی به‌جای این sentinel، استثنای دامنه با زمینه داریم.

## C08 — ترتیب handlerها از خاص به عام
کلاس FileNotFoundError زیرمجموعهٔ OSError است. نخستین handler منطبق انتخاب می‌شود؛ بنابراین handler عمومی‌تر باید پایین‌تر باشد.

In [ ]:
try:
    raise FileNotFoundError("missing input")
except FileNotFoundError:
    print("missing file")
except OSError:
    print("other io failure")

این مثال خطا را عمداً می‌سازد تا به فایل‌سیستم وابسته نباشد. ترتیب معکوس از نظر نحوی معتبر است، اما پیام اختصاصی فایل گمشده هرگز انتخاب نمی‌شود. Exception نیز همهٔ زیرکلاس‌های BaseException را پوشش نمی‌دهد.

## C09 — else برای مسیر موفق
else فقط زمانی اجرا می‌شود که try بدون استثنا تمام شده باشد. این ابزار دامنهٔ گرفتن خطا را محدودتر و خواناتر می‌کند.

In [ ]:
for text in ("4", "bad"):
    try:
        count = int(text)
    except ValueError:
        print("invalid count")
    else:
        print("accepted", count)

استثنایی که در خود else ایجاد شود توسط exceptهای همین try گرفته نمی‌شود. این تفاوت مفید است: خرابی قالب‌بندی یا محاسبه را با شکست تبدیل ورودی یکی نمی‌کنیم.

## C10 — finally برای پاک‌سازی، نه ساخت پاسخ
در مسیر مدیریت‌شدهٔ پایتون، finally چه پس از موفقیت و چه هنگام انتشار استثنا اجرا می‌شود. اثر آن را با فهرست رویدادها می‌بینیم.

In [ ]:
events = []
try:
    try:
        events.append("start")
        raise ValueError("invalid input")
    finally:
        events.append("cleanup")
except ValueError:
    events.append("handled")
print(events)

این تضمین را به قطع برق، kill اجباری فرایند یا خرابی مفسر تعمیم ندهید. return یا استثنای تازه در finally می‌تواند نتیجه یا خطای قبلی را بپوشاند؛ برای آزادسازی فایل معمولاً with روشن‌تر است.

## C11 — اعتبارسنجی صریح با raise
مرز ورودی باید حتی با فعال‌بودن بهینه‌سازی هم برقرار بماند. bool را نیز جدا می‌کنیم، چون isinstance(True, int) درست است ولی تعداد قراردادی ما نیست.

In [ ]:
def require_quantity(value):
    if type(value) is not int:
        raise TypeError("quantity must be an integer")
    if value <= 0:
        raise ValueError("quantity must be positive")
    return value

for value in (3, 0, True):
    try:
        print(require_quantity(value))
    except (TypeError, ValueError) as exc:
        print(type(exc).__name__)

انتخاب نوع استثنا بخشی از API است. در این تابع آموزشی نوع و دامنه را جدا کرده‌ایم؛ در پروژهٔ گزارش، هر دو زیر یک DataContractError شناخته‌شده به مرز برنامه می‌رسند. هیچ‌کدام را با assert برای ورودی کاربر جایگزین نمی‌کنیم.

## C12 — استثنای اختصاصی دامنه
استثنای اختصاصی به مرز برنامه اجازه می‌دهد خطای داده را از باگ محاسبه تشخیص دهد. برای این نیاز، وراثت کوچک کافی است و به سلسله‌مراتب پیچیده نیاز نداریم.

In [ ]:
class DataContractError(ValueError):
    pass

try:
    raise DataContractError("invalid_quantity record=2")
except DataContractError as exc:
    print(type(exc).__name__)
    print(str(exc))

زیرکلاس‌بودن از ValueError یک تصمیم قراردادی است. صرف نام اختصاصی، معنای خطا را تضمین نمی‌کند؛ فقط خطاهای قابل‌انتظار ورودی را به این نوع تبدیل می‌کنیم و خطاهای برنامه‌نویسی را زیر آن پنهان نمی‌کنیم.

## C13 — افزودن زمینه با زنجیرهٔ استثنا
شمارهٔ رکورد برای اصلاح داده لازم است، اما نباید علت اولیهٔ تبدیل عدد را دور بریزیم. raise ... from ... این رابطه را آشکار می‌کند.

In [ ]:
class DataContractError(ValueError):
    pass

def parse_quantity(text):
    try:
        return int(text)
    except ValueError as exc:
        raise DataContractError("invalid_integer record=2 field=quantity") from exc

try:
    parse_quantity("bad")
except DataContractError as exc:
    print(str(exc))
    print(type(exc.__cause__).__name__)

پیام بیرونی متن خام را چاپ نمی‌کند، اما traceback اشکال‌زدایی ممکن است پیام علت اولیه را نشان دهد. بنابراین --debug برای محیط کنترل‌شده است؛ حذف داده از پیام عمومی به معنی پاک‌بودن همهٔ لاگ‌ها و زنجیره‌ها نیست.

## C14 — انتشار دوباره با raise تنها
گاهی لایهٔ میانی فقط باید رویداد را ثبت و همان شکست را منتشر کند. بازگرداندن صفر در این نقطه، نتیجهٔ نامعتبر را ظاهراً موفق جلوه می‌دهد.

In [ ]:
events = []
def middle():
    try:
        int("bad")
    except ValueError:
        events.append("observed")
        raise

try:
    middle()
except ValueError:
    print(events)
    print("still failed")

raise تنها در handler، استثنای فعال را دوباره منتشر می‌کند. raise exc می‌تواند قاب اضافیِ محل raise را به traceback اضافه کند. ثبت چندبارهٔ یک traceback در همهٔ لایه‌ها معمولاً لاگ را شلوغ می‌کند؛ مالک ثبت نهایی را تعیین کنید.

## C15 — بستن منبع در مسیر شکست
مدیریت منبع به مسیر موفق محدود نیست. این مثال با فایل موقت، بسته‌شدن handle پس از استثنا را بدون اتکا به زمان جمع‌آوری زباله نشان می‌دهد.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "sample.txt"
    path.write_text("ready", encoding="ascii")
    try:
        with path.open(encoding="ascii") as source:
            raise ValueError("stop reading")
    except ValueError:
        print(source.closed)

with تضمین می‌کند مسیر خروج معمول یا استثنایی از بلوک، متد خروج منبع را فراخوانی کند. این با تضمین موفقیت پاک‌سازی در هر خرابی سیستم یکی نیست. برای مولد فایل‌دار فصل دهم، مالکِ مصرف از closing استفاده می‌کند.

## C16 — assert برای فرض داخلی
assert ابزار بیان فرض توسعه‌دهنده است؛ اگر شرط برقرار نباشد AssertionError ایجاد می‌کند. اینجا دادهٔ داخلی و شناخته‌شده را بررسی می‌کنیم.

In [ ]:
gross = 300
refund = 100
retained = 200
assert gross - refund == retained, "balance mismatch"
print("balance checked")

پیام باید فرض نقض‌شده را توضیح دهد، نه دادهٔ حساس را. assert قرارداد امنیتی، اعتبارسنجی ورودی کاربر یا جایگزین مدیریت استثنا نیست؛ مثال بعد دلیل اجرایی این مرز را نشان می‌دهد.

## C17 — چرا assert مرز ورودی نیست
همان برنامهٔ کوچک را در دو مفسر فرزند اجرا می‌کنیم. تغییر پرچم بهینه‌سازی، شاهدی واقعی و مستقل از تنظیم کرنل فعلی به دست می‌دهد.

In [ ]:
import subprocess
import sys

code = "assert False, 'blocked'; print('continued')"
normal = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
optimized = subprocess.run([sys.executable, "-O", "-c", code], capture_output=True, text=True)
print(normal.returncode != 0)
print(optimized.returncode)
print(optimized.stdout.strip())

با -O دستور assert حذف می‌شود؛ به عبارت شرط و پیام آن نیز برای اثر جانبی تکیه نکنید. برای کنترل‌هایی که باید در انتشار برقرار بمانند، if و raise بنویسید. متدهای assertEqual و assertRaises در unittest دستور assert زبان نیستند.

## C18 — الگوی آماده‌سازی، اقدام، بررسی
آزمون را با سه مرحلهٔ Arrange، Act و Assert می‌خوانیم. مقدار مورد انتظار را از محاسبهٔ دستی تعیین می‌کنیم، نه با فراخوانی دوبارهٔ تابع تحت آزمون.

In [ ]:
def retained(quantity, price, returned):
    return (quantity - returned) * price

quantity, price, returned = 3, 100, 1
expected = 200
actual = retained(quantity, price, returned)
print(actual == expected)

این print برای نمایش ساختار است؛ در مجموعه‌آزمون باید شکست به test runner اعلام شود، نه اینکه فقط False چاپ شود. در مثال بعد runner واقعی داریم. اگر oracle همان فرمول اشتباه را کپی کند، آزمون هم همان اشتباه را تأیید می‌کند.

## C19 — اولین TestCase واقعی
unittest همراه پایتون است. نام متد آزمون با test شروع می‌شود و runner نتیجه را ثبت می‌کند؛ خروجی زمان‌دار runner را در بافر نگه می‌داریم تا مثال خروجی پایدار داشته باشد.

In [ ]:
import io
import unittest

def retained(quantity, price, returned):
    return (quantity - returned) * price

class RetainedTests(unittest.TestCase):
    def test_partial_return(self):
        self.assertEqual(retained(3, 100, 1), 200)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(RetainedTests)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print(result.testsRun)
print(result.wasSuccessful())

در فایل واقعی، unittest.main یا discovery نتیجه را به ترمینال و کد خروج منتقل می‌کند. درون دفترچه بدون مدیریت argv از unittest.main استفاده نکنید؛ پارامترهای کرنل می‌توانند با پارامترهای runner تداخل کنند.

## C20 — آزمودن شکست مورد انتظار
فقط تست پاسخ سالم کافی نیست؛ تابع باید ورودی نامعتبر را با نوع و پیام قراردادی رد کند. بدنهٔ assertRaisesRegex را تا حد امکان کوچک می‌گیریم.

In [ ]:
import io
import unittest

def require_positive(value):
    if value <= 0:
        raise ValueError("quantity must be positive")
    return value

class ValidationTests(unittest.TestCase):
    def test_zero_is_rejected(self):
        with self.assertRaisesRegex(ValueError, "positive"):
            require_positive(0)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(ValidationTests)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print(result.wasSuccessful())

Regex کل پیام را لزوماً برابر نمی‌گیرد. برای پیام دارای شناسهٔ پایدار، می‌توان ویژگی exc.code را جدا آزمود؛ به متن طولانی و نسخه‌وابستهٔ پیام داخلی Python تکیه نکنید. گرفتن هر Exception در آزمون بیش از حد سست است.

## C21 — مرزها با subTest
ورودی مرزی از دل قرارداد می‌آید: بدون برگشت، برگشت جزئی، برگشت کامل و قیمت صفر. هر ردیف یک انتظار مستقل دارد.

In [ ]:
import io
import unittest

class BoundaryTests(unittest.TestCase):
    def test_cases(self):
        cases = [(3, 100, 0, 300), (3, 100, 1, 200),
                 (3, 100, 3, 0), (3, 0, 1, 0)]
        for quantity, price, returned, expected in cases:
            with self.subTest(quantity=quantity, returned=returned):
                self.assertEqual((quantity - returned) * price, expected)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(BoundaryTests)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print(result.testsRun)
print(result.wasSuccessful())

testsRun تعداد متد آزمون را می‌شمارد؛ چهار subTest را نباید چهار آزمون مستقل در گزارش runner اعلام کرد. subTest باعث می‌شود شکست یک مورد، مانع بررسی باقی ردیف‌های همان حلقه نشود.

## C22 — fixture تازه و پاک‌سازی ثبت‌شده
آزمون خوب از فایل شخصی یا پوشهٔ اجرای فعلی مستقل است. برای منابع موقت، پاک‌سازی را بلافاصله پس از ساخت ثبت می‌کنیم.

In [ ]:
import io
from pathlib import Path
from tempfile import TemporaryDirectory
import unittest

class FileTests(unittest.TestCase):
    def setUp(self):
        self.directory = TemporaryDirectory()
        self.addCleanup(self.directory.cleanup)
        self.path = Path(self.directory.name) / "data.txt"

    def test_roundtrip(self):
        self.path.write_text("ready", encoding="ascii")
        self.assertEqual(self.path.read_text(encoding="ascii"), "ready")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(FileTests)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print(result.wasSuccessful())

در unittest، setUp برای هر متد آزمون اجرا می‌شود. اگر پس از ثبت addCleanup بخشی از setUp شکست بخورد، پاک‌سازی ثبت‌شده همچنان اجرا می‌شود؛ tearDown برای setUp ناموفق اجرا نمی‌شود. این تفاوت برای مدیریت منابع مهم است.

## C23 — آزمون بازگشت برای یک باگ کشف‌شده
قبل از اصلاح، آزمون باید شکست معنی‌دار نشان دهد؛ بعد از اصلاح باید موفق شود. اینجا یک آزمون یکسان را روی دو پیاده‌سازی اجرا می‌کنیم.

In [ ]:
import io
import unittest

def check_function(function):
    class RegressionTest(unittest.TestCase):
        def test_partial_return(self):
            self.assertEqual(function(3, 100, 1), 200)
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(RegressionTest)
    result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
    return result.wasSuccessful()

print(check_function(lambda q, p, r: q * p - r))
print(check_function(lambda q, p, r: (q - r) * p))

شکست اول عمدی و مهارشده است و کل سلول را شکست‌خورده نمی‌کند. اگر هر دو True می‌شدند، آزمون قدرت آشکارسازی این باگ را نداشت. یک باگ، یک آزمون بازگشت قابل‌فهم؛ بعداً می‌توان دامنهٔ حالت‌ها را گسترش داد.

## C24 — مقایسهٔ تقریبی با معیار روشن
عددهای ممیز شناور نمایش دقیق همهٔ اعشار را ندارند. برای سهم یا نسبت، دامنه و تلورانس را صریح می‌کنیم؛ مبلغ صحیح تومان همچنان با برابری دقیق سنجیده می‌شود.

In [ ]:
import math
value = 0.1 + 0.2
print(value == 0.3)
print(math.isclose(value, 0.3, rel_tol=1e-12, abs_tol=1e-12))

isclose برابری ریاضی نیست. آزمون نزدیکی باید علاوه بر واحد، مقیاس و معنای خطا را مشخص کند. بزرگ‌گرفتن بی‌دلیل تلورانس می‌تواند یک باگ واقعی را مخفی کند؛ قرار نیست هر اختلافی را به گردن float بیندازیم.

## C25 — NaN و بی‌نهایت، بدون استثنا
بعضی ورودی‌های غیرمتناهی به‌راحتی ساخته می‌شوند و ممکن است خطای زمان اجرا ندهند. برای دامنه‌ای که مقدار متناهی می‌خواهد، باید آن را صریح بررسی کرد.

In [ ]:
import math
for value in (1.5, float("nan"), float("inf")):
    print(math.isfinite(value))
print(float("nan") == float("nan"))

NaN حتی با خودش برابر نیست. این رفتار را با missing قراردادی یا عدد صفر یکی نکنید. پروژهٔ پولی از integer استفاده می‌کند؛ بحث این مثال برای سهم‌های float و فصل‌های محاسبات عددی بعدی است.

## C26 — نزدیک صفر، تلورانس مطلق
اگر مقدار مرجع صفر باشد، تلورانس نسبی به‌تنهایی معمولاً کمکی نمی‌کند. تلورانس مطلق باید در واحد کمیت تعریف شود.

In [ ]:
import math
value = 1e-10
print(math.isclose(value, 0.0, rel_tol=1e-6))
print(math.isclose(value, 0.0, rel_tol=1e-6, abs_tol=1e-9))

فرمول معیار، اختلاف مطلق را با بزرگ‌ترِ تلورانس مطلق و تلورانس نسبیِ مقیاس مقایسه می‌کند. ۱e-9 انتخاب آموزشی این مثال است، نه توصیهٔ عمومی برای همهٔ دستگاه‌ها و واحدها.

## C27 — آزمون رابطه‌ای، افزون بر پاسخ ثابت
گاهی رابطه‌ای بین چند اجرا داریم: دو برابرشدن قیمت باید مبلغ را دو برابر کند و جابه‌جایی ترتیب رکوردها نباید جمع را عوض کند.

In [ ]:
def total(rows):
    return sum((q - r) * p for q, p, r in rows)

rows = [(3, 100, 1), (2, 50, 0)]
scaled = [(q, 2 * p, r) for q, p, r in rows]
print(total(rows) == total(list(reversed(rows))))
print(total(scaled) == 2 * total(rows))

این‌ها آزمون‌های رابطه‌ای یا metamorphic هستند، نه اثبات صحت. تابعی که همیشه صفر بدهد هر دو را پاس می‌کند؛ بنابراین در کنار رابطه‌ها باید oracle عددی مستقل هم داشته باشیم. تولید تصادفی یا Hypothesis در این مثال استفاده نشده است.

## C28 — تزریق وابستگی برای خطای قابل‌بازتولید
برای آزمودن خطای خواندن، لازم نیست واقعاً دیسک را خراب کنیم. تابع یک reader دریافت می‌کند و در آزمون نسخهٔ ساختگیِ کنترل‌شده به آن می‌دهیم.

In [ ]:
def read_and_total(reader):
    return sum(reader())

def failed_reader():
    raise OSError("simulated read failure")

try:
    read_and_total(failed_reader)
except OSError:
    print("io failure propagated")

این تزریق خطا شاهد مسیر برخورد با شکست است، نه آزمون همهٔ رفتارهای فایل‌سیستم. در پروژهٔ نهایی علاوه بر mock مرزی، CSV واقعی در پوشهٔ موقت هم خوانده می‌شود تا آزمون واحد و یکپارچه مکمل هم باشند.

## C29 — ثبت رویداد بدون آلودگی تنظیمات عمومی
logging به خروجی گزارش تعلق ندارد. در این مثال logger مستقل و بافر محلی داریم؛ بنابراین اجرای دوبارهٔ سلول، handler تکراری به root logger اضافه نمی‌کند.

In [ ]:
import io
import logging
buffer = io.StringIO()
logger = logging.Logger("chapter11.example")
handler = logging.StreamHandler(buffer)
handler.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
logger.addHandler(handler)
try:
    logger.warning("invalid record=%s field=%s", 2, "quantity")
finally:
    logger.removeHandler(handler)
    handler.close()
print(buffer.getvalue(), end="")

پارامترگذاری پیام از قالب‌بندی زودهنگام جلوگیری می‌کند، ولی راهکار امنیت داده نیست. logging.exception در handler traceback را هم ثبت می‌کند و ممکن است متن خام علت را افشا کند. سیاست دسترسی و نگه‌داری لاگ، جدا از صحت کد است.

## C30 — آزمون یکپارچهٔ کوچک از فایل تا عدد
در پایان، مرز ورودی و محاسبه را با یک فایل موقت به هم متصل می‌کنیم. دادهٔ کوچک اجازه می‌دهد انتظار را مستقل و دستی حساب کنیم.

In [ ]:
import csv
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "orders.csv"
    path.write_text("quantity,price,returned\n3,100,1\n2,50,0\n", encoding="ascii")
    with path.open(encoding="ascii", newline="") as source:
        rows = csv.DictReader(source)
        total = sum((int(row["quantity"]) - int(row["returned"]))
                    * int(row["price"]) for row in rows)
    print(total)
    print(total == 300)

این schema سه‌ستونی فقط برای مثال مستقل است؛ جای قرارداد شش‌ستونی پروژه را نمی‌گیرد. خواندن موفق این فایل شاهد اعتبار تاریخ، یکتایی شناسه و دامنهٔ برگشت‌ها نیست؛ پروژهٔ کارگاهی این کنترل‌ها را جدا اعمال می‌کند.